
# Download Cholec80 to Google Drive

Downloads, verifies, and extracts the **Cholec80** dataset
([Twinanda et al., IEEE TMI 2017](https://doi.org/10.1109/TMI.2016.2593957)) from
CAMMA's authenticated mirror:

```
https://s3.unistra.fr/camma_public/datasets/cholec80/cholec80.zip
```

This requires CAMMA access credentials (requested from the CAMMA group,
`camma.dataset@gmail.com`) -- you'll be prompted for them below via `getpass`,
so they're never echoed to the cell output or saved to disk.

**License**: non-commercial research use only. Don't redistribute the raw
data or commit it to any repo.

## What this notebook does

1. Mounts Google Drive (on Colab) and points every path at
   `MyDrive/endofm-benchmark/data/cholec80` -- kept alongside the existing
   Kvasir-Capsule benchmark data in the same project folder.
2. If `cholec80.zip` already exists there, **verifies it's not corrupted**
   (per-member CRC check via `zipfile.testzip()`) before trusting it --
   a truncated/interrupted prior download is deleted and re-fetched rather
   than silently extracted.
3. Downloads with **resume support** (`Range` header against a `.part` file)
   since this is a large, multi-hour download over a Colab session that can
   disconnect -- a dropped connection loses only the unwritten tail, not the
   whole file.
4. Re-validates after download, then extracts (skipped if already extracted).

## Storage

CAMMA's own tooling repo (`TF-Cholec80`) reports **~166GB** for the zip and
**~85GB** after extraction -- budget Drive space accordingly. The actual
download size is read from the server's `Content-Length` header at run time
rather than assumed.


## 0. Setup: paths and packages

On Colab: **Runtime -> Run all**. Mounts Drive and installs `requests`/`tqdm`
if missing (both usually ship with Colab already). Running locally, this uses
`E:/endofm-benchmark` directly.

In [ ]:

import sys, subprocess
from pathlib import Path

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/drive/MyDrive/endofm-benchmark")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "requests", "tqdm"], check=True)
else:
    PROJECT_ROOT = Path(r"E:/endofm-benchmark")

DATA_DIR = PROJECT_ROOT / "data" / "cholec80"
ZIP_PATH = DATA_DIR / "cholec80.zip"
PART_PATH = DATA_DIR / "cholec80.zip.part"
VERIFIED_MARKER = DATA_DIR / "cholec80.zip.verified"
EXTRACT_DIR = DATA_DIR / "raw"

DATA_DIR.mkdir(parents=True, exist_ok=True)
print("Data dir:", DATA_DIR)


## 1. Credentials

Enter your CAMMA username/password when prompted. `getpass` masks the input
so it never appears in the cell, the notebook's saved output, or anywhere on
disk -- it only lives in memory for this session.

In [ ]:

import getpass

CAMMA_USERNAME = getpass.getpass("CAMMA username: ")
CAMMA_PASSWORD = getpass.getpass("CAMMA password: ")
print("Credentials captured (not displayed).")


## 2. Integrity check helper

`zipfile.testzip()` reads and CRC-checks every member of the archive -- the
only reliable corruption check available here, since CAMMA doesn't publish a
checksum for this file. It's a full read pass, so it's slow on a 166GB file;
a `.verified` marker (recording the zip's size) is written after a successful
check so re-running this notebook doesn't re-validate a file that hasn't
changed.

In [ ]:

import zipfile

def is_verified(zip_path, marker_path):
    if not (zip_path.exists() and marker_path.exists()):
        return False
    return marker_path.read_text().strip() == str(zip_path.stat().st_size)

def check_integrity(zip_path):
    if not zipfile.is_zipfile(zip_path):
        return False
    try:
        with zipfile.ZipFile(zip_path) as zf:
            bad_file = zf.testzip()
        return bad_file is None
    except Exception as e:
        print("Integrity check raised:", e)
        return False

def ensure_valid_zip(zip_path, marker_path):
    # Returns True if zip_path exists and is a valid, complete zip.
    if not zip_path.exists():
        return False
    if is_verified(zip_path, marker_path):
        print(f"Already verified (size unchanged): {zip_path} "
              f"({zip_path.stat().st_size / 1e9:.2f} GB)")
        return True
    print(f"Found {zip_path}, running integrity check (full read pass, may take a while)...")
    if check_integrity(zip_path):
        marker_path.write_text(str(zip_path.stat().st_size))
        print("Integrity check passed.")
        return True
    print("Integrity check FAILED -- file is corrupted or incomplete. Deleting for re-download.")
    zip_path.unlink()
    marker_path.unlink(missing_ok=True)
    return False


## 3. Download (resumable, authenticated)

Streams to a `.part` file so a dropped Colab session loses only the unwritten
tail, not the whole download. If the server doesn't honor the `Range`
request (returns `200` instead of `206`), falls back to a clean restart.

In [ ]:

import requests
from tqdm.auto import tqdm

CHOLEC80_URL = "https://s3.unistra.fr/camma_public/datasets/cholec80/cholec80.zip"

def download_cholec80(url, final_path, part_path, username, password, chunk_mb=8):
    auth = (username, password)
    resume_pos = part_path.stat().st_size if part_path.exists() else 0
    headers = {"Range": f"bytes={resume_pos}-"} if resume_pos else {}

    with requests.get(url, auth=auth, headers=headers, stream=True, timeout=60) as r:
        if r.status_code == 401:
            raise RuntimeError("401 Unauthorized -- check your CAMMA username/password and re-run the credentials cell.")
        if resume_pos and r.status_code == 200:
            print("Server doesn't support resuming this download; restarting from 0.")
            resume_pos = 0
            part_path.unlink()
        r.raise_for_status()

        total = resume_pos + int(r.headers.get("content-length", 0))
        mode = "ab" if resume_pos else "wb"
        with open(part_path, mode) as f, tqdm(
            total=total, initial=resume_pos, unit="B", unit_scale=True, unit_divisor=1024,
            desc="cholec80.zip",
        ) as pbar:
            for chunk in r.iter_content(chunk_size=chunk_mb * 1024 * 1024):
                f.write(chunk)
                pbar.update(len(chunk))

    part_path.rename(final_path)
    print("Download complete:", final_path)

if ensure_valid_zip(ZIP_PATH, VERIFIED_MARKER):
    print("Skipping download -- valid archive already present.")
else:
    download_cholec80(CHOLEC80_URL, ZIP_PATH, PART_PATH, CAMMA_USERNAME, CAMMA_PASSWORD)
    if not ensure_valid_zip(ZIP_PATH, VERIFIED_MARKER):
        raise RuntimeError(
            "Downloaded file failed integrity check after a fresh download -- "
            "likely a transfer error or server-side issue. Re-run this cell to retry."
        )


## 4. Extraction

Skipped if `data/cholec80/raw/` already has content, so re-running this
notebook after a full extraction is cheap.

In [ ]:

if EXTRACT_DIR.exists() and any(EXTRACT_DIR.rglob("*")):
    print("Already extracted:", EXTRACT_DIR)
else:
    EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH) as zf:
        members = zf.infolist()
        for member in tqdm(members, desc="Extracting"):
            zf.extract(member, EXTRACT_DIR)
    print("Extraction complete:", EXTRACT_DIR)


## 5. Sanity check

In [ ]:

video_files = list(EXTRACT_DIR.rglob("*.mp4")) + list(EXTRACT_DIR.rglob("*.avi"))
total_bytes = sum(p.stat().st_size for p in video_files)
print(f"Video files found: {len(video_files)}")
print(f"Total size on disk: {total_bytes / 1e9:.1f} GB")
print("Sample paths:", [str(p.relative_to(EXTRACT_DIR)) for p in video_files[:5]])


## Notes / caveats

- **Endo-FM pretraining overlap**: Endo-FM's pretraining corpus includes
  `CholecTriplet`, which is built from a 45-video subset of Cholec80
  (`CholecT45`). If you're benchmarking Endo-FM's frozen backbone on Cholec80
  phase/tool recognition, exclude those ~45 videos from your held-out
  test/eval split, or the result partly measures "performance on data the
  backbone already saw unlabeled during pretraining," not generalization.
  The exact `CholecT45` video-ID list still needs to be pulled from
  `CAMMA-public/cholect50` to build that exclusion precisely -- not done in
  this notebook, which only downloads the raw data.
- This notebook does not extract frames or build any benchmark pipeline --
  that's a follow-up step, analogous to `extract_frames.py` /
  `benchmark_endofm_kvasir_capsule.ipynb` for Kvasir-Capsule.
